# Step 16 — Agentic Workflow Design Patterns

**Optional.** [Step 14](step_14_multi_agent_seq.ipynb) is the last required individual exercise notebook — this one, like [Step 15](step_15_multi_agent_hierarchical.ipynb), is worth exploring once the required notebooks are done. Anthropic's engineering guide, [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents), names five recurring patterns for composing LLM calls into a workflow: prompt chaining, routing, parallelization, orchestrator-workers, and evaluator-optimizer. You've already built two of them without the label — chaining in [Step 05](step_05_chain_prompting.ipynb)/[Step 14](step_14_multi_agent_seq.ipynb), orchestrator-workers in [Step 15](step_15_multi_agent_hierarchical.ipynb). This notebook walks through all five with a working CrewAI implementation each, plus two more you've already met as plain LLM calls — chain-of-thought ([Step 06](step_06_chain_of_thought.ipynb)) and tree-of-thought ([Step 07](step_07_tree_of_thought.ipynb)) — now rebuilt properly inside the framework.

## Learning objective

By the end of this notebook, you will:

- Know Anthropic's five workflow patterns and which CrewAI mechanism implements each one
- Have run a `ConditionalTask`-based router that skips every branch except the one a classifier task selected
- Have run two tasks in true parallel via `async_execution=True`, and seen a later task automatically wait for both
- Have built a `guardrail` callable that runs its own one-off `Crew` to judge another task's output — CrewAI's version of evaluator-optimizer
- Have used `Agent(reasoning=True)` for a built-in chain-of-thought pass, and rebuilt Step 07's tree-of-thought with genuinely parallel agents instead of one simulated in a single prompt
- Be able to judge, for your own project, which pattern (if any) actually fits the problem you're solving — see `REPORT.md`'s Section 5.1

## Prerequisites

- [Step 08 — Introduction to CrewAI](step_08_intro_to_crewai.ipynb) completed — this notebook assumes you know `Agent`/`Task`/`Crew` and won't repeat the full parameter reference
- [Step 14 — Multi-Agent (Sequential)](step_14_multi_agent_seq.ipynb) and [Step 15 — Multi-Agent (Hierarchical)](step_15_multi_agent_hierarchical.ipynb) recommended, not required — patterns 1 and 6 below reuse mechanisms those notebooks cover in depth
- The same `.env` setup as the previous steps

## Background

> Anthropic. (2024). *Building Effective Agents*. https://www.anthropic.com/engineering/building-effective-agents

The article's central argument is easy to miss under the pattern names: **most tasks don't need the most complex pattern, or even an autonomous agent at all.** Start with the simplest composition that solves the problem, and only reach for more structure — routing, parallel branches, a manager, a feedback loop — when the task genuinely demands it. None of the patterns below is a default choice. Chain-of-thought and tree-of-thought (Patterns 2–3) sit right after chaining because they're the other two prompting techniques from Steps 06–07, not because they fit Anthropic's own ordering — the remaining five (Patterns 1, 4–7) are Anthropic's, roughly ordered by how much control they hand to the LLM.

## How this works

Seven patterns, seven CrewAI mechanisms — each gets its own section below with a working example:

| Pattern | What it does | CrewAI mechanism |
| --- | --- | --- |
| Prompt Chaining | Decompose into ordered steps, each processing the last | `Task(context=[prior_task])` |
| Chain of Thought | Reason step by step before answering, within one task | `Agent(reasoning=True)` |
| Tree of Thought | Explore several independent reasoning paths, then compare | Parallelization (below) + a synthesis `Task` |
| Routing | Classify, then send to one specialized branch | `crewai.tasks.conditional_task.ConditionalTask` |
| Parallelization | Run independent subtasks concurrently, then combine | `Task(async_execution=True)` |
| Orchestrator-Workers | A manager delegates each piece of work at runtime | `Process.hierarchical` + `manager_llm` |
| Evaluator-Optimizer | One LLM judges another's output, retries with feedback | `Task(guardrail=<callable>)` |

Each section below is self-contained and doesn't depend on state from the others — run them in order, or jump around.

### Pattern 1 — Prompt Chaining

Decompose a task into a fixed sequence of steps, where each step's output becomes the next step's input. `Task(context=[prior_task])` is CrewAI's native mechanism — you've already used it in [Step 14](step_14_multi_agent_seq.ipynb) to hand the Researcher's output to the Analyst.

In [1]:
import os
import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

writer = Agent(
    role="Content Writer",
    goal="Write clear, well-structured content",
    backstory="You are an experienced technical writer who always outlines before writing.",
    llm=llm,
    verbose=True,
)

# ── Step 1: outline, gated by the check above ─────────────────────────────────
outline_task = Task(
    description="Write a 3-bullet outline for a short blog post about why prompt chaining helps LLM output quality.",
    expected_output="A 3-bullet outline, one line per bullet.",
    agent=writer,
)

# ── Step 2: expand — context=[outline_task] chains step 1's output into step 2 ─
draft_task = Task(
    description="Using the outline above, write the full blog post — one short paragraph per bullet.",
    expected_output="A complete blog post, one paragraph per outline bullet.",
    agent=writer,
    context=[outline_task],
)

chain_crew = Crew(agents=[writer], tasks=[outline_task, draft_task], process=Process.sequential, tracing=True, verbose=True)

start = time.time()
result = chain_crew.kickoff()
elapsed = time.time() - start

print("=== Outline (gated) ===")
print(outline_task.output.raw)
print("\n=== Final draft (chained from outline) ===")
print(result.raw)

# ── KPIs — see Step 17 for a reusable version of this ─────────────────────────
print(f"\n[KPIs] {elapsed:.2f}s | {result.token_usage.total_tokens} tokens "
      f"({result.token_usage.prompt_tokens} prompt / {result.token_usage.completion_tokens} completion)")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  bad65dee-e71a-4f9c-a3e7-f863c64d320f                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Write a 3-bullet outline for a short blog post about why prompt chaining helps LLM output quality.       │
│  ID: 88a963fd-0cb3-42ef-8696-6e8ac2eed96d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Task: Write a 3-bullet outline for a short blog post about why prompt chaining helps LLM output quality.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  * **Deconstructing Complex Tasks:** Explain how breaking a large request into smaller, sequential steps        │
│  prevents LLMs from hallucinating or losing focus.                                                              │
│  * **Managing Context and Constraints:** Detail how prompt chaining allows developers to pass intermediate      │
│  outputs as focused instructions for subsequent steps to ensure accuracy.                                       │
│  * **Iterative Refinement and Error Handling:** Describe how chaining enables a modular workflow where each     │
│  step validates the previous output to improve overall consistency.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Write a 3-bullet outline for a short blog post about why prompt chaining helps LLM output quality.             │
│  Agent:                                                                                                         │
│  Content Writer                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Task: Using the outline above, write the full blog post — one short paragraph per bullet.                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  When you ask an LLM to tackle a massive, multi-faceted task in a single prompt, the model often struggles to   │
│  maintain coherence, leading to hallucinations or tangential errors. By deconstructing complex goals into       │
│  smaller, logical steps, you effectively reduce the "cognitive load" on the model, allowing it to focus its     │
│  attention on one specific objective at a time. This modular approach ensures that each phase of the process    │
│  receives the necessary detail and precision, resulting in a more reliable and grounded final output.           │
│                                                                                                                 │
│  Prompt chaining significantly enhances quality by passing the intermediate results of one step directly into   │
│  the next as structured context. Instead of forcing the LLM to juggle conflicting instructions all at once,     │
│  chaining allows you to layer constraints systematically, ensuring that specific requirements are met at every  │
│  stage of the pipeline. This flow keeps the model’s reasoning tightly aligned with your goals, as each          │
│  subsequent prompt acts as a narrowed-down frame of reference based on the preceding work.                      │
│                                                                                                                 │
│  Finally, chaining creates an iterative workflow that acts as a built-in safety net for error handling and      │
│  refinement. Because each step generates a distinct output, developers can inspect, validate, or even           │
│  re-process individual segments of the chain if the results aren't up to standard before the process proceeds   │
│  further. This modular oversight ensures that inconsistencies are caught early, ultimately leading to a more    │
│  consistent, professional, and higher-quality final result than a single-shot prompt could ever achieve.        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the outline above, write the full blog post — one short paragraph per bullet.                            │
│  Agent:                                                                                                         │
│  Content Writer                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  bad65dee-e71a-4f9c-a3e7-f863c64d320f                                                                           │
│  Final Output: When you ask an LLM to tackle a massive, multi-faceted task in a single prompt, the model often  │
│  struggles to maintain coherence, leading to hallucinations or tangential errors. By deconstructing complex     │
│  goals into smaller, logical steps, you effectively reduce the "cognitive load" on the model, allowing it to    │
│  focus its attention on one specific objective at a time. This modular approach ensures that each phase of the  │
│  process receives the necessary detail and precision, resulting in a more reliable and grounded final output.   │
│                                                                                                                 │
│  Prompt chaining significantly enhances quality by passing the intermediate results of one step directly into   │
│  the next as structured context. Instead of forcing the LLM to juggle conflicting instructions all at once,     │
│  chaining allows you to layer constraints systematically, ensuring that specific requirements are met at every  │
│  stage of the pipeline. This flow keeps the model’s reasoning tightly aligned with your goals, as each          │
│  subsequent prompt acts as a narrowed-down frame of reference based on the preceding work.                      │
│                                                                                                                 │
│  Finally, chaining creates an iterative workflow that acts as a built-in safety net for error handling and      │
│  refinement. Because each step generates a distinct output, developers can inspect, validate, or even           │
│  re-process individual segments of the chain if the results aren't up to standard before the process proceeds   │
│  further. This modular oversight ensures that inconsistencies are caught early, ultimately leading to a more    │
│  consistent, professional, and higher-quality final result than a single-shot prompt could ever achieve.        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

=== Outline (gated) ===
* **Deconstructing Complex Tasks:** Explain how breaking a large request into smaller, sequential steps prevents LLMs from hallucinating or losing focus.
* **Managing Context and Constraints:** Detail how prompt chaining allows developers to pass intermediate outputs as focused instructions for subsequent steps to ensure accuracy.
* **Iterative Refinement and Error Handling:** Describe how chaining enables a modular workflow where each step validates the previous output to improve overall consistency.

=== Final draft (chained from outline) ===
When you ask an LLM to tackle a massive, multi-faceted task in a single prompt, the model often struggles to maintain coherence, leading to hallucinations or tangential errors. By deconstructing complex goals into smaller, logical steps, you effectively reduce the "cognitive load" on the model, allowing it to focus its attention on one specific objective at a time. This modular approach ensures that each phase of the proc

### Pattern 2 — Chain of Thought

Get the model to reason step by step before committing to a final answer, instead of jumping straight to a conclusion — the technique from [Step 06](step_06_chain_of_thought.ipynb), now inside the framework instead of a plain `llm.call()`. CrewAI's native mechanism is `Agent(reasoning=True)`: a plan-and-refine pass that runs before the agent starts the task, capped by `max_reasoning_attempts`. The simpler alternative needs no framework feature at all — just say "think step by step" directly in the `Task.description` or `Agent.backstory`, exactly like Step 06 did.

In [2]:
import os
import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

# ── reasoning=True is CrewAI's built-in CoT: a plan-and-refine pass runs
# before the agent starts the task itself, visible in the verbose log. ───────
solver = Agent(
    role="Word Problem Solver",
    goal="Solve multi-step word problems correctly, not just plausibly",
    backstory="You never skip a step, even when the final answer seems obvious.",
    llm=llm,
    reasoning=True,
    max_reasoning_attempts=2,
    verbose=True,
)

solve_task = Task(
    description=(
        "A store had 120 apples. It sold 35% of them in the morning, then sold "
        "18 more in the afternoon. Some apples were then thrown away because they "
        "went bad — a third of what remained after the afternoon sale. How many "
        "apples does the store have left?"
    ),
    expected_output="The final number of apples left, with the calculation shown.",
    agent=solver,
)

cot_crew = Crew(agents=[solver], tasks=[solve_task], process=Process.sequential, verbose=True)

start = time.time()
result = cot_crew.kickoff()
elapsed = time.time() - start

print(result.raw)

# ── KPIs — see Step 17 for a reusable version of this ─────────────────────────
print(f"\n[KPIs] {elapsed:.2f}s | {result.token_usage.total_tokens} tokens "
      f"({result.token_usage.prompt_tokens} prompt / {result.token_usage.completion_tokens} completion)")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  fb4b5c93-9bbe-4eb4-88f5-e1a33e491299                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: A store had 120 apples. It sold 35% of them in the morning, then sold 18 more in the afternoon. Some     │
│  apples were then thrown away because they went bad — a third of what remained after the afternoon sale. How    │
│  many apples does the store have left?                                                                          │
│  ID: 71e71e6f-19be-48f3-8504-0fa6a9e34e47                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🧠 Reasoning ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Reasoning Started                                                                                              │
│  Attempt: 1                                                                                                     │
│  Status: Thinking...                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Reasoning Complete ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Reasoning Completed                                                                                            │
│  Status: Ready                                                                                                  │
│  Plan: ### Plan for Solving the Apple Inventory Problem                                                         │
│                                                                                                                 │
│  **1. Understanding of the Task**                                                                               │
│  From my professional perspective, this is a sequential inventory problem. The task requires calculating the    │
│  remaining inventory after a series of distinct operations (percentage reduction, fixed unit reduction, and     │
│  fractional disposal). My role is to ensure that each stage of the calculation is isolated, verified, and       │
│  applied correctly to the remaining balance to avoid compounding errors.                                        │
│                                                                                                                 │
│  **2. Key Steps**                                                                                               │
│  *   **S...                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Word Problem Solver                                                                                     │
│                                                                                                                 │
│  Task: A store had 120 apples. It sold 35% of them in the morning, then sold 18 more in the afternoon. Some     │
│  apples were then thrown away because they went bad — a third of what remained after the afternoon sale. How    │
│  many apples does the store have left?                                                                          │
│                                                                                                                 │
│  Reasoning Plan:                                                                                                │
│  ### Plan for Solving the Apple Inventory Problem                                                               │
│                                                                                                                 │
│  **1. Understanding of the Task**                                                                               │
│  From my professional perspective, this is a sequential inventory problem. The task requires calculating the    │
│  remaining inventory after a series of distinct operations (percentage reduction, fixed unit reduction, and     │
│  fractional disposal). My role is to ensure that each stage of the calculation is isolated, verified, and       │
│  applied correctly to the remaining balance to avoid compounding errors.                                        │
│                                                                                                                 │
│  **2. Key Steps**                                                                                               │
│  *   **Step 1:** Calculate the number of apples sold in the morning (35% of 120).                               │
│  *   **Step 2:** Determine the remainder after the morning sale (Initial - Morning Sales).                      │
│  *   **Step 3:** Subtract the afternoon sales (18 apples) from the result of Step 2.                            │
│  *   **Step 4:** Calculate the number of apples thrown away (one-third of the remainder from Step 3).           │
│  *   **Step 5:** Calculate the final inventory by subtracting the disposed apples from the Step 3 remainder.    │
│                                                                                                                 │
│  **3. Approach to Challenges**                                                                                  │
│  The primary challenge is ensuring that the "third of what remained" is calculated based on the precise         │
│  inventory level *after* the afternoon sales, not on the original total. I will mitigate this risk by clearly   │
│  documenting each intermediate value and performing a final verification of the subtraction logic at every      │
│  step.                                                                                                          │
│                                                                                                                 │
│  **4. Strategic Use of Tools**                                                                                  │
│  I will use basic arithmetic operators provided by the environment to ensure precision.                         │
│  *   *Step 1:* `120 * 0.35`                                                                                     │
│  *   *Step 2:* `120 - (Result of Step 1)`                                                                       │
│  *   *Step 3:* `(Result of Step 2) - 18`               

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Word Problem Solver                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **                                                                                                             │
│  The store has **40** apples left.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  A store had 120 apples. It sold 35% of them in the morning, then sold 18 more in the afternoon. Some apples    │
│  were then thrown away because they went bad — a third of what remained after the afternoon sale. How many      │
│  apples does the store have left?                                                                               │
│                                                                                                                 │
│  Reasoning Plan:                                                                                                │
│  ### Plan for Solving the Apple Inventory Problem                                                               │
│                                                                                                                 │
│  **1. Understanding of the Task**                                                                               │
│  From my professional perspective, this is a sequential inventory problem. The task requires calculating the    │
│  remaining inventory after a series of distinct operations (percentage reduction, fixed unit reduction, and     │
│  fractional disposal). My role is to ensure that each stage of the calculation is isolated, verified, and       │
│  applied correctly to the remaining balance to avoid compounding errors.                                        │
│                                                                                                                 │
│  **2. Key Steps**                                                                                               │
│  *   **Step 1:** Calculate the number of apples sold in the morning (35% of 120).                               │
│  *   **Step 2:** Determine the remainder after the morning sale (Initial - Morning Sales).                      │
│  *   **Step 3:** Subtract the afternoon sales (18 apples) from the result of Step 2.                            │
│  *   **Step 4:** Calculate the number of apples thrown away (one-third of the remainder from Step 3).           │
│  *   **Step 5:** Calculate the final inventory by subtracting the disposed apples from the Step 3 remainder.    │
│                                                                                                                 │
│  **3. Approach to Challenges**                                                                                  │
│  The primary challenge is ensuring that the "third of what remained" is calculated based on the precise         │
│  inventory level *after* the afternoon sales, not on the original total. I will mitigate this risk by clearly   │
│  documenting each intermediate value and performing a final verification of the subtraction logic at every      │
│  step.                                                                                                          │
│                                                                                                                 │
│  **4. Strategic Use of Tools**                                                                                  │
│  I will use basic arithmetic operators provided by the environment to ensure precision.                         │
│  *   *Step 1:* `120 * 0.35`                                                                                     │
│  *   *Step 2:* `120 - (Result of Step 1)`                                                                       │
│  *   *Step 3:* `(Result of Step 2) - 18`               

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  fb4b5c93-9bbe-4eb4-88f5-e1a33e491299                                                                           │
│  Final Output: **                                                                                               │
│  The store has **40** apples left.                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**
The store has **40** apples left.

[KPIs] 3.69s | 1980 tokens (1128 prompt / 852 completion)


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Pattern 3 — Tree of Thought

Explore several independent reasoning paths, then compare and pick or merge the best one — the technique from [Step 07](step_07_tree_of_thought.ipynb). That notebook's own Shortcomings flagged a real weakness: simulating multiple "experts" in a single prompt still means one model reasoning about all paths, which can converge on a shared blind spot. CrewAI fixes that directly: reuse Pattern 5's parallelization (below) for genuinely separate `Agent`s exploring each path independently, then a synthesis `Task` (`context=[...]`) compares them — real tree-of-thought, not a simulated one.

In [3]:
import os
import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

question = "Should a small campus café accept cryptocurrency payments?"

optimist = Agent(role="Optimist Analyst", goal="Argue the strongest case in favor", backstory="You focus on upside and opportunity.", llm=llm, verbose=True)
skeptic = Agent(role="Skeptic Analyst", goal="Argue the strongest case against", backstory="You focus on risk and what could go wrong.", llm=llm, verbose=True)
pragmatist = Agent(role="Pragmatist Analyst", goal="Focus on practical, operational concerns only", backstory="You care about what's actually feasible to implement.", llm=llm, verbose=True)
judge = Agent(role="Decision Judge", goal="Weigh multiple independent analyses into one recommendation", backstory="You synthesize without just picking a favorite.", llm=llm, verbose=True)

# ── Three genuinely independent reasoning paths, run in true parallel ────────
path_optimist = Task(description=question, expected_output="A short case in favor, 3 points max.", agent=optimist, async_execution=True)
path_skeptic = Task(description=question, expected_output="A short case against, 3 points max.", agent=skeptic, async_execution=True)
path_pragmatist = Task(description=question, expected_output="A short list of practical implementation concerns.", agent=pragmatist, async_execution=True)

# ── Synchronous — waits for all three paths, then compares them ──────────────
judge_task = Task(
    description="Compare the three independent analyses above and give one final, justified recommendation.",
    expected_output="A final recommendation with a short justification that references where the three paths agreed or disagreed.",
    agent=judge,
    context=[path_optimist, path_skeptic, path_pragmatist],
)

tot_crew = Crew(
    agents=[optimist, skeptic, pragmatist, judge],
    tasks=[path_optimist, path_skeptic, path_pragmatist, judge_task],
    process=Process.sequential,
    verbose=True,
)

start = time.time()
result = tot_crew.kickoff()
elapsed = time.time() - start

print("=== Optimist path ===")
print(path_optimist.output.raw)
print("\n=== Skeptic path ===")
print(path_skeptic.output.raw)
print("\n=== Pragmatist path ===")
print(path_pragmatist.output.raw)
print("\n=== Judge's final recommendation ===")
print(result.raw)

# ── KPIs — see Step 17 for a reusable version of this. Four full agent runs
# (three paths + the judge) show up here as one combined token total. ────────
print(f"\n[KPIs] {elapsed:.2f}s | {result.token_usage.total_tokens} tokens "
      f"({result.token_usage.prompt_tokens} prompt / {result.token_usage.completion_tokens} completion)")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  5b94efd5-cb62-4c3e-a2f0-74bd27ad0a20                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Should a small campus café accept cryptocurrency payments?                                               │
│  ID: 914fbd4b-8fbd-4fc7-abf9-a9ffc62fd340                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Should a small campus café accept cryptocurrency payments?                                               │
│  ID: b5e61bdb-63d9-40f4-a9fe-e472064a7a82                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Should a small campus café accept cryptocurrency payments?                                               │
│  ID: 3750e193-d3a6-4319-94cd-91d57523b864                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Optimist Analyst                                                                                        │
│                                                                                                                 │
│  Task: Should a small campus café accept cryptocurrency payments?                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Decision Judge                                                                                          │
│                                                                                                                 │
│  Task: Compare the three independent analyses above and give one final, justified recommendation.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Decision Judge                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Final Recommendation: Conditional Pilot Program via Third-Party Payment Processor                          │
│                                                                                                                 │
│  After synthesizing the three independent analyses, I recommend that the café **avoid direct self-custody of    │
│  cryptocurrency but pursue a trial period using a specialized third-party crypto-payment processor.**           │
│                                                                                                                 │
│  #### Justification                                                                                             │
│  The three analyses present a tension between marketing innovation and operational reality. All sides agree     │
│  that technology-driven payment methods possess a significant "cool factor" that appeals to a student           │
│  demographic, serving as a powerful differentiator for a small business. However, they also reach a consensus   │
│  that the technical and financial risks—specifically volatility, network latency, and accounting                │
│  complexity—are significant enough to threaten a café’s razor-thin profit margins if handled manually.          │
│                                                                                                                 │
│  The "Against" and "Operational" analyses highlight that attempting to handle crypto assets directly            │
│  (self-custody) is untenable for a small business due to transaction friction and tax reporting burdens.        │
│  However, the "Pro" analysis identifies a genuine opportunity to capture the student market.                    │
│                                                                                                                 │
│  **To bridge these positions, the café should adopt a strategy that delegates the risk:**                       │
│  1.  **Use a Third-Party Processor:** By utilizing a service that instantly converts cryptocurrency into fiat   │
│  currency (USD) at the point of sale, the café negates the volatility and margin erosion issues cited in the    │
│  "Against" argument, while still providing the requested "tech-forward" payment option.                         │
│  2.  **Mitigate Operational Friction:** The "Operational" analysis correctly notes that transaction latency     │
│  and gas fees are deal-breakers. A professional processor will manage these through secondary layer networks    │
│  (like Lightning Network for Bitcoin) or stablecoin settlements, ensuring the transaction is as fast as a       │
│  traditional credit card tap.                                                                                   │
│  3.  **Address Administrative Burden:** Modern payment gateways offer integrated tax and accounting exports,    │
│  which addresses the primary administrative hurdle raised in the "Against" analysis.                            │
│                                                                                                                 │
│  **Conclusion:** Implementing a direct, DIY crypto-wallet system is ill-advised and potentially destructive to  │
│  your margins. However, integrating a third-party gateway allows you to capture the innovative "third space"    │
│  brand identity desired by students while offloading th

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Compare the three independent analyses above and give one final, justified recommendation.                     │
│  Agent:                                                                                                         │
│  Decision Judge                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

=== Optimist path ===
Adopting cryptocurrency payments at a small campus café is a strategic move that positions the business as a forward-thinking hub for the next generation of consumers. Here is the case for implementation:

1. **Attracting Tech-Savvy Students:** By accepting crypto, the café creates an immediate draw for students interested in fintech and decentralized finance. This signals that the business is modern, innovative, and aligned with the values of a digitally native demographic, effectively turning the café into a preferred "third space" for the tech-conscious crowd.

2. **Differentiating from Competitors:** In a competitive campus landscape, offering a unique payment method serves as a powerful marketing differentiator. It transforms a simple transaction into a novel experience, increasing word-of-mouth visibility and giving the café a distinct edge over traditional competitors who rely solely on legacy payment systems.

3. **Lowering Transaction Costs and Barriers:*

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  5b94efd5-cb62-4c3e-a2f0-74bd27ad0a20                                                                           │
│  Final Output: ### Final Recommendation: Conditional Pilot Program via Third-Party Payment Processor            │
│                                                                                                                 │
│  After synthesizing the three independent analyses, I recommend that the café **avoid direct self-custody of    │
│  cryptocurrency but pursue a trial period using a specialized third-party crypto-payment processor.**           │
│                                                                                                                 │
│  #### Justification                                                                                             │
│  The three analyses present a tension between marketing innovation and operational reality. All sides agree     │
│  that technology-driven payment methods possess a significant "cool factor" that appeals to a student           │
│  demographic, serving as a powerful differentiator for a small business. However, they also reach a consensus   │
│  that the technical and financial risks—specifically volatility, network latency, and accounting                │
│  complexity—are significant enough to threaten a café’s razor-thin profit margins if handled manually.          │
│                                                                                                                 │
│  The "Against" and "Operational" analyses highlight that attempting to handle crypto assets directly            │
│  (self-custody) is untenable for a small business due to transaction friction and tax reporting burdens.        │
│  However, the "Pro" analysis identifies a genuine opportunity to capture the student market.                    │
│                                                                                                                 │
│  **To bridge these positions, the café should adopt a strategy that delegates the risk:**                       │
│  1.  **Use a Third-Party Processor:** By utilizing a service that instantly converts cryptocurrency into fiat   │
│  currency (USD) at the point of sale, the café negates the volatility and margin erosion issues cited in the    │
│  "Against" argument, while still providing the requested "tech-forward" payment option.                         │
│  2.  **Mitigate Operational Friction:** The "Operational" analysis correctly notes that transaction latency     │
│  and gas fees are deal-breakers. A professional processor will manage these through secondary layer networks    │
│  (like Lightning Network for Bitcoin) or stablecoin settlements, ensuring the transaction is as fast as a       │
│  traditional credit card tap.                                                                                   │
│  3.  **Address Administrative Burden:** Modern payment gateways offer integrated tax and accounting exports,    │
│  which addresses the primary administrative hurdle raised in the "Against" analysis.                            │
│                                                                                                                 │
│  **Conclusion:** Implementing a direct, DIY crypto-wal

### Pattern 4 — Routing

Classify an input, then direct it to exactly one specialized follow-up task. `crewai.tasks.conditional_task.ConditionalTask` is CrewAI's native routing primitive: it's skipped or executed based on a `condition: Callable[[TaskOutput], bool]` evaluated against a *prior* task's output. It runs inside a plain `Process.sequential` `Crew` — no Flows needed, though CrewAI's own `Flow`/`@router` decorator (out of scope for this course, see [Step 08](step_08_intro_to_crewai.ipynb)'s Background) is the more general tool if you need routing across more than one categorical gate.

In [4]:
import os
import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tasks.conditional_task import ConditionalTask

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

router_agent = Agent(
    role="Request Classifier",
    goal="Classify incoming support requests as billing or technical",
    backstory="You quickly and accurately categorize customer messages.",
    llm=llm,
    verbose=True,
)
billing_agent = Agent(
    role="Billing Support Specialist",
    goal="Resolve billing questions clearly and politely",
    backstory="You know refunds, invoices, and subscription plans inside out.",
    llm=llm,
    verbose=True,
)
technical_agent = Agent(
    role="Technical Support Specialist",
    goal="Resolve technical questions with precise, actionable steps",
    backstory="You debug software issues methodically.",
    llm=llm,
    verbose=True,
)

user_message = "I was charged twice this month for my subscription, can you help?"

# ── Classify first ─────────────────────────────────────────────────────────
classify_task = Task(
    description=f"Classify this customer message as exactly one word, 'billing' or 'technical':\n\n{user_message}",
    expected_output="A single word: billing or technical. Nothing else.",
    agent=router_agent,
)

# ── Only the branch whose condition matches the classifier's output runs ─────
billing_task = ConditionalTask(
    condition=lambda output: "billing" in output.raw.lower(),
    description=f"Write a helpful reply to this billing question:\n\n{user_message}",
    expected_output="A helpful, polite reply to the customer's billing question.",
    agent=billing_agent,
)
technical_task = ConditionalTask(
    condition=lambda output: "technical" in output.raw.lower(),
    description=f"Write a helpful reply to this technical question:\n\n{user_message}",
    expected_output="A helpful, precise reply to the customer's technical question.",
    agent=technical_agent,
)

routing_crew = Crew(
    agents=[router_agent, billing_agent, technical_agent],
    tasks=[classify_task, billing_task, technical_task],
    process=Process.sequential,
    tracing=True,
    verbose=True,
)

start = time.time()
result = routing_crew.kickoff()
elapsed = time.time() - start

print("Classification:", classify_task.output.raw)
print("\nFinal reply (only the matching branch ran):")
print(result.raw)

# ── KPIs — see Step 17 for a reusable version of this. The skipped branch's
# agent never gets called, so its tokens don't show up here at all. ──────────
print(f"\n[KPIs] {elapsed:.2f}s | {result.token_usage.total_tokens} tokens "
      f"({result.token_usage.prompt_tokens} prompt / {result.token_usage.completion_tokens} completion)")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  cf8c59ae-2954-4a66-8023-ac5d72369f11                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Classify this customer message as exactly one word, 'billing' or 'technical':                            │
│                                                                                                                 │
│  I was charged twice this month for my subscription, can you help?                                              │
│  ID: 721480c5-27b7-4fcf-95aa-9b6e89c0270a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Request Classifier                                                                                      │
│                                                                                                                 │
│  Task: Classify this customer message as exactly one word, 'billing' or 'technical':                            │
│                                                                                                                 │
│  I was charged twice this month for my subscription, can you help?                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Request Classifier                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  billing                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Classify this customer message as exactly one word, 'billing' or 'technical':                                  │
│                                                                                                                 │
│  I was charged twice this month for my subscription, can you help?                                              │
│  Agent:                                                                                                         │
│  Request Classifier                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Billing Support Specialist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Regarding your recent subscription charges                                                            │
│                                                                                                                 │
│  Hello,                                                                                                         │
│                                                                                                                 │
│  Thank you for reaching out to us. I am very sorry to hear that you have been charged twice for your            │
│  subscription this month; I understand how concerning it is to see unexpected charges on your statement.        │
│                                                                                                                 │
│  I would be happy to look into this for you immediately to resolve the issue. To help me locate your account    │
│  and identify the cause of the duplicate charge, could you please provide the following details?                │
│                                                                                                                 │
│  *   The email address associated with your subscription account.                                               │
│  *   The transaction dates and amounts shown on your statement.                                                 │
│  *   The last four digits of the payment method used.                                                           │
│                                                                                                                 │
│  Once I have this information, I will investigate the billing history, verify the duplicate charge, and         │
│  initiate a refund for the extra payment if it was processed in error.                                          │
│                                                                                                                 │
│  Rest assured that we will get this corrected for you as quickly as possible. I look forward to hearing from    │
│  you.                                                                                                           │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  Billing Support Specialist                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Write a helpful reply to this billing question:                                                                │
│                                                                                                                 │
│  I was charged twice this month for my subscription, can you help?                                              │
│  Agent:                                                                                                         │
│  Billing Support Specialist                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[2026-07-29 17:50:41][DEBUG]: Skipping conditional task: Write a helpful reply to this technical question:

I was charged twice this month for my subscription, can you help?


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  cf8c59ae-2954-4a66-8023-ac5d72369f11                                                                           │
│  Final Output: Subject: Regarding your recent subscription charges                                              │
│                                                                                                                 │
│  Hello,                                                                                                         │
│                                                                                                                 │
│  Thank you for reaching out to us. I am very sorry to hear that you have been charged twice for your            │
│  subscription this month; I understand how concerning it is to see unexpected charges on your statement.        │
│                                                                                                                 │
│  I would be happy to look into this for you immediately to resolve the issue. To help me locate your account    │
│  and identify the cause of the duplicate charge, could you please provide the following details?                │
│                                                                                                                 │
│  *   The email address associated with your subscription account.                                               │
│  *   The transaction dates and amounts shown on your statement.                                                 │
│  *   The last four digits of the payment method used.                                                           │
│                                                                                                                 │
│  Once I have this information, I will investigate the billing history, verify the duplicate charge, and         │
│  initiate a refund for the extra payment if it was processed in error.                                          │
│                                                                                                                 │
│  Rest assured that we will get this corrected for you as quickly as possible. I look forward to hearing from    │
│  you.                                                                                                           │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  Billing Support Specialist                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Classification: billing

Final reply (only the matching branch ran):
Subject: Regarding your recent subscription charges

Hello,

Thank you for reaching out to us. I am very sorry to hear that you have been charged twice for your subscription this month; I understand how concerning it is to see unexpected charges on your statement.

I would be happy to look into this for you immediately to resolve the issue. To help me locate your account and identify the cause of the duplicate charge, could you please provide the following details?

*   The email address associated with your subscription account.
*   The transaction dates and amounts shown on your statement.
*   The last four digits of the payment method used.

Once I have this information, I will investigate the billing history, verify the duplicate charge, and initiate a refund for the extra payment if it was processed in error. 

Rest assured that we will get this corrected for you as quickly as possible. I look forward to hearing 

### Pattern 5 — Parallelization (Sectioning)

Run independent subtasks concurrently instead of one after another, then combine their outputs. `Task(async_execution=True)` runs a task in a background thread; the *next synchronous task* automatically waits for every pending async task and receives their combined output through `context=[...]` — verified directly against CrewAI's `Crew.kickoff()` source for this course, not just assumed from the docs.

In [5]:
import os
import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

pros_agent = Agent(role="Pros Analyst", goal="Identify the strongest advantages of a given approach", backstory="You focus only on upside.", llm=llm, verbose=True)
cons_agent = Agent(role="Cons Analyst", goal="Identify the strongest drawbacks of a given approach", backstory="You focus only on downside.", llm=llm, verbose=True)
synth_agent = Agent(role="Decision Synthesizer", goal="Combine pros and cons into a balanced recommendation", backstory="You weigh trade-offs fairly.", llm=llm, verbose=True)

topic = "using a multi-agent CrewAI system instead of a single LLM call"

# ── Both run concurrently — neither waits on the other ───────────────────────
pros_task = Task(
    description=f"List the top 3 advantages of {topic}.",
    expected_output="3 bullet points, advantages only.",
    agent=pros_agent,
    async_execution=True,
)
cons_task = Task(
    description=f"List the top 3 drawbacks of {topic}.",
    expected_output="3 bullet points, drawbacks only.",
    agent=cons_agent,
    async_execution=True,
)

# ── Synchronous — Crew automatically waits for both async tasks above first ──
synth_task = Task(
    description="Using the pros and cons gathered above, write a short, balanced recommendation.",
    expected_output="A short paragraph weighing the pros and cons into a recommendation.",
    agent=synth_agent,
    context=[pros_task, cons_task],
)

parallel_crew = Crew(
    agents=[pros_agent, cons_agent, synth_agent],
    tasks=[pros_task, cons_task, synth_task],
    process=Process.sequential,
    tracing=True,
    verbose=True,
)

start = time.time()
result = parallel_crew.kickoff()
elapsed = time.time() - start

print("=== Pros (ran in parallel) ===")
print(pros_task.output.raw)
print("\n=== Cons (ran in parallel) ===")
print(cons_task.output.raw)
print("\n=== Synthesis (waited for both) ===")
print(result.raw)

# ── KPIs — see Step 17 for a reusable version of this. Compare `elapsed` here
# against Pattern 1's — two of these three calls ran concurrently, not one
# after another, so the wall-clock time doesn't just add up per task. ────────
print(f"\n[KPIs] {elapsed:.2f}s | {result.token_usage.total_tokens} tokens "
      f"({result.token_usage.prompt_tokens} prompt / {result.token_usage.completion_tokens} completion)")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  1c40ec84-a389-404e-8c99-566350b34356                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: List the top 3 advantages of using a multi-agent CrewAI system instead of a single LLM call.             │
│  ID: 46c64f4e-60da-4c10-90cb-e7643e441c71                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: List the top 3 drawbacks of using a multi-agent CrewAI system instead of a single LLM call.              │
│  ID: 64682536-7778-44c7-99e0-dc72728ffe4e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Pros Analyst                                                                                            │
│                                                                                                                 │
│  Task: List the top 3 advantages of using a multi-agent CrewAI system instead of a single LLM call.             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Cons Analyst                                                                                            │
│                                                                                                                 │
│  Task: List the top 3 drawbacks of using a multi-agent CrewAI system instead of a single LLM call.              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Decision Synthesizer                                                                                    │
│                                                                                                                 │
│  Task: Using the pros and cons gathered above, write a short, balanced recommendation.                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Decision Synthesizer                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Adopting a CrewAI multi-agent framework offers a transformative advantage for complex, multi-faceted projects  │
│  by leveraging specialized role-based expertise and internal verification mechanisms to ensure high-quality,    │
│  coherent outcomes. However, these benefits are countered by significant trade-offs in operational efficiency,  │
│  specifically regarding increased latency, higher token consumption, and the inherent difficulty of debugging   │
│  a non-linear, multi-agent pipeline. Therefore, the recommendation is to reserve CrewAI for high-stakes,        │
│  sophisticated tasks where precision and structural depth are the primary requirements, while favoring          │
│  simpler, monolithic prompting for routine or time-sensitive workflows where minimizing latency and             │
│  architectural complexity is the priority.                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the pros and cons gathered above, write a short, balanced recommendation.                                │
│  Agent:                                                                                                         │
│  Decision Synthesizer                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  1c40ec84-a389-404e-8c99-566350b34356                                                                           │
│  Final Output: Adopting a CrewAI multi-agent framework offers a transformative advantage for complex,           │
│  multi-faceted projects by leveraging specialized role-based expertise and internal verification mechanisms to  │
│  ensure high-quality, coherent outcomes. However, these benefits are countered by significant trade-offs in     │
│  operational efficiency, specifically regarding increased latency, higher token consumption, and the inherent   │
│  difficulty of debugging a non-linear, multi-agent pipeline. Therefore, the recommendation is to reserve        │
│  CrewAI for high-stakes, sophisticated tasks where precision and structural depth are the primary               │
│  requirements, while favoring simpler, monolithic prompting for routine or time-sensitive workflows where       │
│  minimizing latency and architectural complexity is the priority.                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

=== Pros (ran in parallel) ===
* **Specialized Expertise through Role-Based Decomposition:** By assigning distinct personas and specific tools to individual agents, a CrewAI system mimics a high-performance team, allowing each agent to focus on deep-level domain execution rather than task-switching, which significantly elevates the quality and precision of complex, multi-faceted outputs.

* **Enhanced Reliability through Iterative Verification:** A multi-agent framework inherently creates a "checks and balances" architecture; one agent can act as an evaluator or critic for another, drastically reducing hallucinations and errors while ensuring that the final output undergoes a rigorous internal refinement process before reaching the user.

* **Superior Handling of Complex Workflows:** Unlike a single LLM call which struggles with long-context degradation and logical coherence on massive tasks, a CrewAI system orchestrates a structured, step-by-step pipeline where agents pass state-depen

### Pattern 6 — Orchestrator-Workers

A central LLM dynamically breaks down a problem and delegates pieces of it to worker LLMs, then the results flow back to it. This is exactly [Step 15](step_15_multi_agent_hierarchical.ipynb)'s `Process.hierarchical`: the manager (built from `manager_llm`, or your own `Agent` passed as `manager_agent`) is the orchestrator, `agents=[...]` are the workers, and tasks get no fixed `agent=` — the manager decides who does what at runtime. See Step 15 for exactly how the delegation mechanism works under the hood (`allow_delegation`, the `Delegate work to coworker` tool); this cell is a second, independent example rather than a repeat of that one.

In [6]:
import os
import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

researcher = Agent(role="Researcher", goal="Gather accurate facts on a given topic", backstory="You are thorough and cite what you find.", llm=llm, verbose=True)
writer = Agent(role="Writer", goal="Turn research into engaging, readable prose", backstory="You write clearly for a general audience.", llm=llm, verbose=True)

# ── No agent= on either task — the manager assigns them at runtime ───────────
research_task = Task(
    description="Research three interesting facts about the Model Context Protocol (MCP).",
    expected_output="Three facts, each 1-2 sentences.",
)
writing_task = Task(
    description="Turn the research findings into a short, engaging paragraph for a newsletter.",
    expected_output="One engaging paragraph, no bullet points.",
)

orchestrator_crew = Crew(
    agents=[researcher, writer],  # the manager's coworkers — the manager itself is not one of them
    tasks=[research_task, writing_task],
    process=Process.hierarchical,
    manager_llm=llm,
    verbose=True,
)

start = time.time()
result = orchestrator_crew.kickoff()
elapsed = time.time() - start

print(result.raw)

# ── KPIs — see Step 17 for a reusable version of this. The manager's own
# reasoning calls are included in this total, on top of both workers'. ───────
print(f"\n[KPIs] {elapsed:.2f}s | {result.token_usage.total_tokens} tokens "
      f"({result.token_usage.prompt_tokens} prompt / {result.token_usage.completion_tokens} completion)")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  22a72373-440f-4d73-9dcb-5d5d3937dc79                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research three interesting facts about the Model Context Protocol (MCP).                                 │
│  ID: 65f07b0d-097c-49de-b35e-45d93da9ddd6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Research three interesting facts about the Model Context Protocol (MCP).                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Find three interesting facts about the Model Context Protocol (MCP) and present them in three  │
│  distinct, 1-2 sentence points.', 'context': 'The task is to research three interesting facts abo...            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Task: Find three interesting facts about the Model Context Protocol (MCP) and present them in three distinct,  │
│  1-2 sentence points.                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Think of the Model Context Protocol (MCP) as the universal "USB-C port" for the AI era, finally solving the    │
│  fragmented data headache that has slowed down developers for years. By providing a standardized way for AI     │
│  models to plug into diverse systems like databases, code repositories, and SaaS platforms, MCP eliminates the  │
│  heavy "integration tax" of building custom connections for every new tool. Because this protocol is an         │
│  open-source standard, developers can build an "MCP Server" just once and know it will work seamlessly across   │
│  any compliant AI client, effectively shattering vendor lock-in and paving the way for a truly modular,         │
│  interconnected ecosystem where your AI can finally access the information it needs, no matter where it lives.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Turn the research findings into a short, engaging paragraph for a newsletter.                                  │
│  Agent:                                                                                                         │
│  Crew Manager                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  22a72373-440f-4d73-9dcb-5d5d3937dc79                                                                           │
│  Final Output: Think of the Model Context Protocol (MCP) as the universal "USB-C port" for the AI era, finally  │
│  solving the fragmented data headache that has slowed down developers for years. By providing a standardized    │
│  way for AI models to plug into diverse systems like databases, code repositories, and SaaS platforms, MCP      │
│  eliminates the heavy "integration tax" of building custom connections for every new tool. Because this         │
│  protocol is an open-source standard, developers can build an "MCP Server" just once and know it will work      │
│  seamlessly across any compliant AI client, effectively shattering vendor lock-in and paving the way for a      │
│  truly modular, interconnected ecosystem where your AI can finally access the information it needs, no matter   │
│  where it lives.                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Think of the Model Context Protocol (MCP) as the universal "USB-C port" for the AI era, finally solving the fragmented data headache that has slowed down developers for years. By providing a standardized way for AI models to plug into diverse systems like databases, code repositories, and SaaS platforms, MCP eliminates the heavy "integration tax" of building custom connections for every new tool. Because this protocol is an open-source standard, developers can build an "MCP Server" just once and know it will work seamlessly across any compliant AI client, effectively shattering vendor lock-in and paving the way for a truly modular, interconnected ecosystem where your AI can finally access the information it needs, no matter where it lives.

[KPIs] 5.94s | 17817 tokens (14961 prompt / 2856 completion)


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Pattern 7 — Evaluator-Optimizer

One LLM generates a response, another evaluates it and feeds back until it passes. CrewAI has a built-in shortcut for this: `Task(guardrail="<plain-English rule>")` — pass a string instead of a function and CrewAI spins up its own internal "Guardrail Agent" to judge the output and retry with feedback.

**That shortcut is broken in the installed CrewAI version (1.9.3) when run inside a Jupyter kernel.** `Agent.kickoff()`'s "auto-async" detection sees Jupyter's already-running event loop and returns an unawaited coroutine instead of a result, so the built-in guardrail crashes with `'coroutine' object has no attribute 'pydantic'`. The cell below builds the same judge-with-an-LLM mechanism by hand, via a normal `guardrail` callable that runs its own one-off `Crew` — this sidesteps the bug, and as a side effect makes the mechanism the string shortcut hides fully explicit.

In [7]:
import os
import time

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from pydantic import BaseModel

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

summarizer = Agent(
    role="Summarizer",
    goal="Summarize text concisely",
    backstory="You write tight, information-dense summaries.",
    llm=llm,
    verbose=True,
)

long_text = (
    "Agentic AI systems differ from simple LLM calls in that they can reason over "
    "multiple steps, call external tools, and adapt their plan based on what they "
    "observe. This makes them more powerful for open-ended tasks, but also more "
    "expensive and harder to predict than a fixed pipeline. Anthropic's engineering "
    "blog argues that most production systems should start with the simplest possible "
    "workflow and only add agentic autonomy where the task genuinely requires it, "
    "since every added degree of freedom trades predictability and cost for flexibility."
)

RULE = "the summary must be exactly one sentence, under 30 words, and must not use the word 'agentic'"

class Verdict(BaseModel):
    valid: bool
    feedback: str | None = None

# ── The evaluator: a second, one-off Crew that judges the first agent's output.
# This is exactly what CrewAI's built-in guardrail="..." does internally — done
# manually here to work around the Jupyter bug described above. ──────────────
def llm_judge(output):
    judge_agent = Agent(
        role="Guardrail Agent",
        goal="Validate the output of the task",
        backstory="You are an expert at validating task output and giving precise, actionable feedback when it fails.",
        llm=llm,
    )
    judge_task = Task(
        description=(
            f"Check whether the following text complies with this rule: '{RULE}'.\n\n"
            f"Text:\n{output.raw}\n\n"
            "If it complies, say so. If not, explain exactly what's wrong."
        ),
        expected_output="A verdict on compliance, with feedback if it fails.",
        agent=judge_agent,
        output_pydantic=Verdict,
    )
    verdict = Crew(agents=[judge_agent], tasks=[judge_task], process=Process.sequential).kickoff().pydantic
    if verdict.valid:
        return True, output.raw
    return False, verdict.feedback or "Output did not comply with the guardrail."

summarize_task = Task(
    description=f"Summarize the following text:\n\n{long_text}",
    expected_output="A summary of the text.",
    agent=summarizer,
    guardrail=llm_judge,
    guardrail_max_retries=3,
)

eval_opt_crew = Crew(agents=[summarizer], tasks=[summarize_task], process=Process.sequential, verbose=True)

start = time.time()
result = eval_opt_crew.kickoff()
elapsed = time.time() - start

print(result.raw)

# ── KPIs — see Step 17 for a reusable version of this. Important gotcha: the
# judge's own kickoff() calls inside llm_judge run on a separate Crew, so their
# tokens are NOT included in eval_opt_crew's result.token_usage below — every
# retry costs real tokens this number doesn't show. ───────────────────────────
print(f"\n[KPIs] {elapsed:.2f}s | {result.token_usage.total_tokens} tokens "
      f"({result.token_usage.prompt_tokens} prompt / {result.token_usage.completion_tokens} completion) "
      f"— excludes the judge's own token usage, see comment above")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  c30d0f1a-2103-469e-b4b2-84d7fe6a3ec2                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Summarize the following text:                                                                            │
│                                                                                                                 │
│  Agentic AI systems differ from simple LLM calls in that they can reason over multiple steps, call external     │
│  tools, and adapt their plan based on what they observe. This makes them more powerful for open-ended tasks,    │
│  but also more expensive and harder to predict than a fixed pipeline. Anthropic's engineering blog argues that  │
│  most production systems should start with the simplest possible workflow and only add agentic autonomy where   │
│  the task genuinely requires it, since every added degree of freedom trades predictability and cost for         │
│  flexibility.                                                                                                   │
│  ID: 45ff13f2-56fb-4770-b59a-edf613246d68                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summarizer                                                                                              │
│                                                                                                                 │
│  Task: Summarize the following text:                                                                            │
│                                                                                                                 │
│  Agentic AI systems differ from simple LLM calls in that they can reason over multiple steps, call external     │
│  tools, and adapt their plan based on what they observe. This makes them more powerful for open-ended tasks,    │
│  but also more expensive and harder to predict than a fixed pipeline. Anthropic's engineering blog argues that  │
│  most production systems should start with the simplest possible workflow and only add agentic autonomy where   │
│  the task genuinely requires it, since every added degree of freedom trades predictability and cost for         │
│  flexibility.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summarizer                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Agentic AI systems offer multi-step reasoning, tool usage, and adaptive planning, making them superior for     │
│  complex tasks but inherently less predictable and more expensive than fixed LLM pipelines. Anthropic           │
│  recommends beginning with the simplest possible workflow, introducing agentic autonomy only when necessary to  │
│  balance flexibility against operational costs and reliability.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name:                                                                                                          │
│  def llm_judge(output):                                                                                         │
│      judge_agent = Agent(                                                                                       │
│    ...                                                                                                          │
│  Status:                                                                                                        │
│  🔄 Evaluating                                                                                                  │
│  Attempt:                                                                                                       │
│  1                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summarizer                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Autonomous software using multi-step reasoning and external tools enhances flexibility for complex tasks, yet  │
│  Anthropic advises prioritizing simple pipelines to maintain cost-efficiency and predictability unless          │
│  increased complexity is strictly required.                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Guardrail  blocked (attempt 2/4), retrying due to: The text is 31 words long, which exceeds the required limit of under 30 words.



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summarizer                                                                                              │
│                                                                                                                 │
│  Task: Summarize the following text:                                                                            │
│                                                                                                                 │
│  Agentic AI systems differ from simple LLM calls in that they can reason over multiple steps, call external     │
│  tools, and adapt their plan based on what they observe. This makes them more powerful for open-ended tasks,    │
│  but also more expensive and harder to predict than a fixed pipeline. Anthropic's engineering blog argues that  │
│  most production systems should start with the simplest possible workflow and only add agentic autonomy where   │
│  the task genuinely requires it, since every added degree of freedom trades predictability and cost for         │
│  flexibility.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summarizer                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Autonomous systems using multi-step reasoning offer greater flexibility for complex tasks, but Anthropic       │
│  recommends prioritizing simple pipelines to optimize predictability and cost unless increased autonomy is      │
│  strictly necessary.                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Autonomous systems using multi-step reasoning offer greater flexibility for complex tasks, but Anthropic recommends prioritizing simple pipelines to optimize predictability and cost unless increased autonomy is strictly necessary.

[KPIs] 4.06s | 3070 tokens (2826 prompt / 244 completion) — excludes the judge's own token usage, see comment above


## Your task

1. **Chaining:** Reword `outline_task`'s description to ask for 5 bullets instead of 3, but leave `exactly_three_bullets` unchanged. Watch the guardrail reject the first attempt and force a retry — that's the gate doing its job.
2. **Chain of Thought:** Set `reasoning=False` and rerun the word problem — does the final answer's correctness actually change for a problem this simple, or does `reasoning=True` mostly just add detail to the verbose log?
3. **Tree of Thought:** Add a fourth path (e.g. a "Student Analyst" focused on the target users) and add it to `judge_task`'s `context=[...]`. Does the judge's recommendation meaningfully shift, or does it still lean on the same two or three paths?
4. **Routing:** Change `user_message` to something that's clearly neither billing nor technical (e.g. "What are your office hours?"). What happens? Is that graceful, or does it expose the limits of a two-branch router?
5. **Parallelization:** Add a third parallel task (e.g. a "Risks Analyst") and add it to `synth_task`'s `context=[...]`. Does the synthesis meaningfully use all three, or does one dominate?
6. **Orchestrator-Workers:** Add a third worker agent (e.g. an "Editor") to `agents=[...]` without assigning it a `Task` directly — does the manager ever bring it in on its own, or does it need an explicit task to be delegated at all?
7. **Evaluator-Optimizer:** Loosen `RULE` to something the summarizer's first attempt will likely already satisfy, and watch `guardrail_max_retries` stay at 0 actual retries in the verbose log. Then tighten it further than the example (e.g. "under 15 words") and see how many retries it takes, if it ever passes at all.
8. For your own team's project: which of these seven patterns, if any, actually fits the problem you're solving? Write your answer as evidence for `REPORT.md`'s Section 5.1 (Suitability) — "none of them, a single agent is enough" is a legitimate answer if that's genuinely true.

## Shortcomings

None of these patterns are CrewAI-specific ideas — they're vocabulary for compositions that show up across every agent framework, and CrewAI happens to have a fairly direct mechanism for each. A few limits worth knowing before reaching for them in your own project:

- **Routing** here is a single categorical gate — one classifier, a fixed set of `ConditionalTask` branches. It doesn't generalize to a large or dynamic set of routes the way a real graph-based router would; CrewAI's `Flow`/`@router` decorator is the tool for that, out of this course's scope.
- **Parallelization** only helps when the subtasks are genuinely independent. If `cons_task` needed to know what `pros_task` found, making them concurrent would silently produce wrong results — there's no error CrewAI can raise for a false independence assumption.
- **Evaluator-Optimizer** here needed a hand-built workaround for a real bug in the installed CrewAI version's string-guardrail shortcut inside Jupyter — a reminder that "the framework has a built-in feature for this" is worth testing before you rely on it, not just reading in the docs.
- **Tree of Thought** costs the most of all seven patterns here — three full agent runs plus a judge, versus one call for chain-of-thought. Reserve it for genuinely ambiguous decisions, not questions with a clean right answer.
- These patterns compose. A real system might route first, then run two of the resulting branches in parallel, each gated by its own evaluator — nothing here demonstrates combining more than one pattern in the same crew.

## Resources for further reading

- [Anthropic — Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) — the source for five of the seven patterns in this notebook
- Wei, J., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. NeurIPS 2022. [arXiv:2201.11903](https://arxiv.org/abs/2201.11903) — the source for Pattern 2, also cited in [Step 06](step_06_chain_of_thought.ipynb)
- Yao, S., et al. (2023). *Tree of Thoughts: Deliberate Problem Solving with Large Language Models*. NeurIPS 2023. [arXiv:2305.10601](https://arxiv.org/abs/2305.10601) — the source for Pattern 3, also cited in [Step 07](step_07_tree_of_thought.ipynb)
- [CrewAI Tasks concept docs](https://docs.crewai.com/en/concepts/tasks) — `context`, `async_execution`, `guardrail`/`guardrails`
- [CrewAI Flows docs](https://docs.crewai.com/en/concepts/flows) — `@router` and conditional branching beyond a single `ConditionalTask` gate
- [CrewAI `Process` concept docs](https://docs.crewai.com/en/concepts/processes) — `sequential` vs. `hierarchical`, covered in depth in [Step 15](step_15_multi_agent_hierarchical.ipynb)

## Stretch goal

Pick the one pattern from this notebook that's most relevant to your own team's project, and rebuild it using your own topic and agents instead of the generic examples above. Then answer honestly: does the pattern earn its added complexity for your specific case, or would the simpler thing from Steps 09/14 have done just as well? That's the exact question `REPORT.md`'s Section 5.1 asks — this is where you'd get the evidence for it.

---

This notebook is optional and not part of what's graded — see [Assignment Overview](../../team_assignment/en/assignment-overview.md) for what's actually required.